# Verify library-boundary-outbound migration

One-shot checklist for after the `library-boundary-outbound` PR merges and the Code Ocean
image is rebuilt. Confirms the three things that could break silently: imports resolve on the
real capsule, `data_loading.load_session_quality_filter` still selects the same sessions it did
before the quality-stats access was rewritten, and re-running `aggregate_tongue_movements` on a
real session reproduces the `out_*` columns already in `all_tongue_movements_04022026.parquet`.

Everything here is read-only — no file is overwritten. Code-Ocean-only, guarded by `IS_CO`; it
has nothing to check locally, since none of the three risks are reachable from
`data/for_local/`. See `TODO.md`, "Define the boundary..." and "Fold outbound metrics...", for
what this closes out.

## 1. Setup

In [ ]:
from pathlib import Path

if Path("/root/capsule").exists():
    SCRATCH = Path("/root/capsule/scratch")
    DATA = Path("/root/capsule/data")
    IS_CO = True
else:
    SCRATCH = None
    DATA = None
    IS_CO = False

if not IS_CO:
    print("Local environment: nothing to verify here. Run this notebook on Code Ocean "
          "after the library PR merges and the image rebuilds.")

## 2. Imports resolve on the real capsule

The four consumers whose import cells changed in the boundary migration. A clean run of this cell is the same check as opening each notebook and running its import cell, without opening four notebooks.

In [ ]:
if IS_CO:
    import importlib
    import aind_dynamic_foraging_behavior_video_analysis as lib
    print("library:", lib.__file__)

    # names that only exist post-migration — ImportError here means the image
    # still has the old checkout
    from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_analysis import (
        TONGUE_QUALITY_STATS_FILENAME, load_tongue_quality_stats, get_quality_summary,
    )
    from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_kinematics_utils import (
        compute_outbound_metrics, filter_timestamps_refractory,
    )
    from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_lickometer_utils import (
        detect_licks, calculate_metrics, calculate_metrics_witheventkeys,
    )
    from aind_dynamic_foraging_behavior_video_analysis.kinematics.video_clip_utils import (
        extract_clips_ffmpeg_encode, extract_clips_ffmpeg_after_reencode,
    )

    import data_loading, build_all_tongue_movements  # noqa: F401 (import-time check only)
    importlib.reload(data_loading)

    print("[ok] all post-migration imports resolve")

## 3. Session quality filter is unchanged

Compares `data_loading.load_session_quality_filter` (now reading `tongue_quality_stats.json` through `load_tongue_quality_stats` / `get_quality_summary`) against a hand-parsed read of the same files. Any mismatch means the accessor rewrite changed which sessions pass.

In [ ]:
if IS_CO:
    import json

    base_dirs = [SCRATCH / "session_analysis_mlk"]

    filtered_session_paths = data_loading.load_session_quality_filter(base_dirs)
    via_library = sorted(p.name for p in filtered_session_paths)

    # independent hand-parsed re-derivation, bypassing the library accessor entirely
    def passes_by_hand(stats_path, coverage_min=90.0, duration50_min=0.06):
        d = json.loads(stats_path.read_text())
        cov = float(d.get("coverage_pct", 0.0))
        dur50 = float(d.get("percentiles", {}).get("duration", {}).get("0.5", 0.0))
        return cov > coverage_min and dur50 > duration50_min

    via_hand = sorted(
        subdir.name
        for base_dir in base_dirs
        for subdir in base_dir.iterdir()
        if subdir.is_dir()
        and (subdir / "tongue_quality_stats.json").exists()
        and passes_by_hand(subdir / "tongue_quality_stats.json")
    )

    print(f"library accessor: {len(via_library)} sessions")
    print(f"hand-parsed:       {len(via_hand)} sessions")
    assert via_library == via_hand, (
        "session set differs — the accessor rewrite changed inclusion:\n"
        f"  library only: {sorted(set(via_library) - set(via_hand))}\n"
        f"  hand only:    {sorted(set(via_hand) - set(via_library))}"
    )
    print("[ok] identical session set")

## 4. `out_*` parity on a real session

Re-runs `aggregate_tongue_movements` (which now calls `compute_outbound_metrics` internally) on one already-processed session's frame-level intermediates, and diffs the result against the `out_*` columns already in that session's `tongue_movs.parquet` — the real-data version of the 400-synthetic-movement check done during development. Exact equality is expected for sessions the current pipeline already produced with the `tongue_movements_all` convention (`out_duration = 0.0`, not NaN, when the endpoint is the first frame).

In [ ]:
if IS_CO:
    import pandas as pd
    import numpy as np
    from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import load_intermediate_data
    from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_kinematics_utils import (
        aggregate_tongue_movements,
    )

    example_session = filtered_session_paths[0]
    print("checking:", example_session.name)

    data = load_intermediate_data(example_session)
    jaw = pd.read_parquet(example_session / "intermediate_data" / "kps_raw_jaw.parquet")

    recomputed = aggregate_tongue_movements(data["kins"], {"jaw": jaw})
    recomputed = recomputed.set_index("movement_id").sort_index()

    existing = data["movs"].set_index("movement_id").sort_index()
    out_cols = ["out_duration", "out_peak_velocity", "out_mean_velocity", "out_total_distance"]

    missing = [c for c in out_cols if c not in existing.columns]
    if missing:
        print(f"[skip] existing tongue_movs.parquet has no {missing} yet — "
              "this session predates the pipeline re-run. Re-run the batch pipeline on this "
              "session (or pick one already produced under the current pipeline) before "
              "trusting this check.")
    else:
        common_ids = recomputed.index.intersection(existing.index)
        lhs = recomputed.loc[common_ids, out_cols]
        rhs = existing.loc[common_ids, out_cols]
        try:
            pd.testing.assert_frame_equal(lhs, rhs, check_exact=False, rtol=1e-6)
            print(f"[ok] out_* columns match exactly on {len(common_ids)} movements")
        except AssertionError as e:
            print("[FAIL] out_* mismatch — do not trust the migrated aggregate_tongue_movements "
                  "until this is resolved:")
            raise

## 5. Summary

In [ ]:
if IS_CO:
    print("All checks passed. Safe to:")
    print("  - archive add_outbound.ipynb once the full pipeline re-run + parity check "
          "(TODO.md item) has been done, not just this one-session check")
    print("  - drop the ImportError fallback note in TODO.md's boundary/outbound items")